In [ ]:
!pip uninstall -y torchcodec

Found existing installation: torchcodec 0.11.0
Uninstalling torchcodec-0.11.0:
  Successfully uninstalled torchcodec-0.11.0


In [ ]:
!pip install torchcodec soundfile

  Using cached torchcodec-0.11.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
Using cached torchcodec-0.11.0-cp311-cp311-win_amd64.whl (1.9 MB)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models
import soundfile as sf
import time

FileNotFoundError: Could not find module 'C:\Users\nitie\AppData\Local\anaconda3\Lib\site-packages\torchaudio\lib\libtorchaudio.pyd' (or one of its dependencies). Try using the full path with constructor syntax.

In [ ]:
class AudioDataset(Dataset):
    def __init__(self, root_dir, max_seconds=3, sample_rate=16000):
        self.data_paths = []
        self.labels = []
        self.target_sr = sample_rate 
        self.max_length = max_seconds * sample_rate 
        
        for label_name, label_idx in [("fake", 0), ("real", 1)]:
            folder_path = os.path.join(root_dir, label_name)
            if os.path.exists(folder_path):
                for file in os.listdir(folder_path):
                    if file.endswith('.wav'):
                        self.data_paths.append(os.path.join(folder_path, file))
                        self.labels.append(label_idx)
                        
        # 변경
        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_mels=128,
            n_fft=1024,
            hop_length=512,
            win_length=1024
            )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
                        
    def __len__(self):
        return len(self.data_paths)
        
    def __getitem__(self, idx):
        wav_path = self.data_paths[idx]
        
        audio_array, sr = sf.read(wav_path)
        
        # 파이썬 배열을 파이토치 텐서로 변환 (모양 맞추기)
        waveform = torch.from_numpy(audio_array).float()
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0) # 모노(1채널)인 경우
        else:
            waveform = waveform.t() # 스테레오(2채널)인 경우
            
        # 2) 주파수(Sample Rate) 동기화
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=self.target_sr)
        
        # 3) 스테레오(2채널) 채널이 있다면 모노(1채널)로 합치기
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        # 4) 길이 맞추기 
        if waveform.shape[1] > self.max_length:
            waveform = waveform[:, :self.max_length] 
        else:
            pad_amount = self.max_length - waveform.shape[1]
            waveform = F.pad(waveform, (0, pad_amount)) 
            
        # 5) 파동(Waveform)을 그림(Mel-Spectrogram)으로 변환
        mel = self.mel_spectrogram(waveform)
        mel_db = self.amplitude_to_db(mel) 
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-9)
        
        label = self.labels[idx]
        return mel_db, label

# 데이터 로더 세팅
audio_dir = "2_Processed_Data/audio_visual"
full_dataset = AudioDataset(root_dir=audio_dir)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# ==========================================
# 3. AI 모델 세팅 (ResNet18 - 1채널 스펙트로그램 개조 버전)
# ==========================================
model = models.resnet18(weights='IMAGENET1K_V1')

# 멜 스펙트로그램 이미지도 흑백 사진처럼 1채널이므로 입력층 개조가 필요합니다.
original_conv1 = model.conv1
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.conv1.weight.data = original_conv1.weight.data.mean(dim=1, keepdim=True)

# 2갈래 출력 (Real/Fake)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

model.load_state_dict(torch.load('best_audio_model.pth'))

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 10
best_acc = 0.0

start_time = time.time()

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 15)

    model.train()  
    running_loss = 0.0
    corrects = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad() 
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1) 
        loss = criterion(outputs, labels)

        loss.backward()  
        optimizer.step() 

        running_loss += loss.item() * inputs.size(0)
        corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = corrects.double() / len(train_dataset)
    print(f'[Train] Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

    model.eval()   
    test_loss = 0.0
    test_corrects = 0

    with torch.no_grad(): 
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * inputs.size(0)
            test_corrects += torch.sum(preds == labels.data)

    test_epoch_loss = test_loss / len(test_dataset)
    test_epoch_acc = test_corrects.double() / len(test_dataset)
    print(f'[Test]  Loss: {test_epoch_loss:.4f} | Acc: {test_epoch_acc:.4f}')

    if test_epoch_acc > best_acc:
        best_acc = test_epoch_acc
        torch.save(model.state_dict(), 'best_audio_model.pth')
        print(" 오디오 모델 최고 성능 갱신! 저장 완료!")
    print()

time_elapsed = time.time() - start_time
print(f' 학습 완료! 총 소요 시간: {time_elapsed // 60:.0f}분 {time_elapsed % 60:.0f}초')
print(f' 최고 테스트 정답률: {best_acc:.4f}')

C:\Users\nitie\AppData\Local\Temp\ipykernel_8076\2773743064.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_audio_model.pth'))


Epoch 1/10
---------------
[Train] Loss: 0.2666 | Acc: 0.9335
[Test]  Loss: 0.3597 | Acc: 0.9176

Epoch 2/10
---------------
[Train] Loss: 0.2388 | Acc: 0.9381
[Test]  Loss: 0.3495 | Acc: 0.9176

Epoch 3/10
---------------
[Train] Loss: 0.2303 | Acc: 0.9400
[Test]  Loss: 0.5672 | Acc: 0.9176

Epoch 4/10
---------------
[Train] Loss: 0.2243 | Acc: 0.9391
[Test]  Loss: 0.4770 | Acc: 0.9176

Epoch 5/10
---------------
[Train] Loss: 0.2117 | Acc: 0.9400
[Test]  Loss: 0.4325 | Acc: 0.9176

Epoch 6/10
---------------
[Train] Loss: 0.2059 | Acc: 0.9391
[Test]  Loss: 0.4931 | Acc: 0.9176

Epoch 7/10
---------------
[Train] Loss: 0.2050 | Acc: 0.9391
[Test]  Loss: 0.5082 | Acc: 0.9176

Epoch 8/10
---------------
[Train] Loss: 0.2017 | Acc: 0.9400
[Test]  Loss: 0.5920 | Acc: 0.9176

Epoch 9/10
---------------
[Train] Loss: 0.1985 | Acc: 0.9400
[Test]  Loss: 0.4539 | Acc: 0.9176

Epoch 10/10
---------------
[Train] Loss: 0.2008 | Acc: 0.9391
[Test]  Loss: 0.5944 | Acc: 0.9176

 학습 완료! 총 소요 시간: 2분

: 0.9438